In [35]:
!pip install langchain langchain-community langchain-core langchain-google-genai langchain-text-splitters faiss-cpu pypdf tiktoken

In [36]:
import os, re, time, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Data Understanding & Preprocessing

Load the PDF and extract text from all pages

In [38]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("/content/intro-to-ml-book.pdf").load()
print("Pages loaded:", len(pages))

Pages loaded: 392


Explore the document structure and text quality

In [39]:
all_text = "\n".join(p.page_content for p in pages)

print("Number of pages:", len(pages))
print("Empty pages (under 20 characters):", sum(1 for p in pages if len(p.page_content.strip()) < 20))
print("Words broken across lines by a hyphen:", len(re.findall(r"\w-\n\w", all_text)))
print("Page footers such as 'Title | 37':", len(re.findall(r"\|\s*\d+\s*$", all_text, flags=re.M)))
print("Book title repeated on pages:", all_text.count("Introduction to Machine Learning with Python"))

Number of pages: 392
Empty pages (under 20 characters): 5
Words broken across lines by a hyphen: 56
Page footers such as 'Title | 37': 179
Book title repeated on pages: 2


Split the text into chunks

Chunk size: 700 tokens (assignment range 500–1000). Overlap: 100 tokens (assignment range 50–150).

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=100,
)
chunks = splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 410


Generate embeddings and store them in FAISS

The chunks are sent in small groups, with a pause between groups, to stay under the free-tier limit. This takes about 10–15 minutes.

In [43]:
from langchain_community.vectorstores import FAISS

vector_store = None
BATCH_SIZE = 20

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embedding)
    else:
        vector_store.add_documents(batch)
    print("Embedded", min(start + BATCH_SIZE, len(chunks)), "of", len(chunks))
    time.sleep(40)

print("Vectors stored in FAISS:", vector_store.index.ntotal)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0, description: Embed content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 21h40m51.516889154s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per

In [ ]:
sample_query = "What is overfitting?"

query_vector = embedding.embed_query(sample_query)
results = vector_store.similarity_search_by_vector(query_vector, k=3)

for doc in results:
    print("Page", doc.metadata["page"] + 1)
    print(doc.page_content[:300], "...")
    print()

Retrieve the top-k chunks for a query

The query is converted into an embedding, and the k most similar chunks are returned.

In [ ]:
def retrieve(question, k=5):
    query_vector = embedding.embed_query(question)            # query -> embedding
    return vector_store.similarity_search_by_vector(query_vector, k=k)   # top-k chunks

Experiment with k = 3, 5, 7

In [ ]:
experiment_query = "What is overfitting?"

for k in [3, 5, 7]:
    docs = retrieve(experiment_query, k=k)
    pages_found = sorted({d.metadata["page"] + 1 for d in docs})
    print(f"k={k}: {len(docs)} chunks, from pages {pages_found}")

Answer Generation (RAG)

The prompt gives the LLM the retrieved chunks as its only context, with each chunk labelled by page number. It tells the LLM to use only that context, to reply with a fixed sentence when the answer is not in the context, and to cite the page for each point. "temperature=0" makes the answers repeatable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a machine learning tutor.
Answer the question using ONLY the context below. Do not use any other knowledge.
If the answer is not in the context, reply exactly: "I couldn't find this in the book."
After each point, cite the page number, for example (p. 42).

Context:
{context}

Question: {question}
""")

chain = prompt | llm | StrOutputParser()

Generate grounded answers (shows the answer, the pages used, and the retrieved context)

In [ ]:
def answer(question, k=5):
    docs = retrieve(question, k=k)
    context = "\n\n".join(f"[p. {d.metadata['page'] + 1}] {d.page_content}" for d in docs)
    response = chain.invoke({"context": context, "question": question})

    print("Question:", question)
    print("Answer:", response)
    print("Pages used:", sorted({d.metadata["page"] + 1 for d in docs}))
    print("\nRetrieved context:")
    for d in docs:
        print(f"  [p. {d.metadata['page'] + 1}] {d.page_content[:150]}...")
    print("-" * 80)

End-to-End RAG Application

Demonstration: multiple queries

In [ ]:
demo_questions = [
    "What is overfitting and how can we detect it?",
    "How does the k-nearest neighbors algorithm make a prediction?",
    "What does one-hot encoding do?",
    "What is cross-validation?",
    "What is the capital of France?",
]

for q in demo_questions:
    answer(q, k=5)


Real-time question answering

Type a question and press Enter. Type "exit" to stop.

In [ ]:
while True:
    question = input("Your question: ")
    if question.strip().lower() == "exit":
        print("Goodbye!")
        break
    answer(question, k=5)